In [1]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Conv2D, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras import layers, models
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.optimizers import Adam
from time import *

import os

import matplotlib.pyplot as plt

print(tf.__version__)

2.17.0


In [2]:
# Muat model MobileNetV2 yang telah dilatih sebelumnya tanpa lapisan atas (top)
base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))

# Membekukan lapisan base model sehingga tidak akan dilatih ulang
base_model.trainable = False

# Menambahkan Global Average Pooling
x = GlobalAveragePooling2D()(base_model.output)

# Menambahkan lapisan Dense dengan aktivasi Softmax untuk klasifikasi 13 kelas
output_layer = Dense(13, activation='softmax')(x)

# Membuat model baru base_model = mobilenetV2(Feature Map) dan Output layer FC menggunakan softmax
model = Model(inputs=base_model.input, outputs=output_layer)
# Menampilkan ringkasan model
model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,274,637 (8.68 MB)

 Trainable params: 16,653 (65.05 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

# Jumlah Layer Pada MobileNetV2

In [3]:
def custom_summary_with_numbers(model):
    print(f"{'No.':<5} {'Layer Name':<25} {'Output Shape':<25} {'Param #':<10}")
    print("=" * 65)
    for i, layer in enumerate(model.layers):
        output_shape = getattr(layer, 'output_shape', 'N/A')  # Cek apakah output_shape tersedia
        print(f"{i+1:<5} {layer.name:<25} {str(output_shape):<25} {layer.count_params():<10}")
    print("=" * 65)
    print(f"Total parameters: {model.count_params()}")

# Menampilkan summary dengan nomor urut
custom_summary_with_numbers(model)


No.   Layer Name                Output Shape              Param #   
1     input_layer               N/A                       0         
2     Conv1                     N/A                       864       
3     bn_Conv1                  N/A                       128       
4     Conv1_relu                N/A                       0         
5     expanded_conv_depthwise   N/A                       288       
6     expanded_conv_depthwise_BN N/A                       128       
7     expanded_conv_depthwise_relu N/A                       0         
8     expanded_conv_project     N/A                       512       
9     expanded_conv_project_BN  N/A                       64        
10    block_1_expand            N/A                       1536      
11    block_1_expand_BN         N/A                       384       
12    block_1_expand_relu       N/A                       0         
13    block_1_pad               N/A                       0         
14    block_1_depthwise       

# Training Model MobileNetV2

In [ ]:
def data_load(data_dir, img_height, img_width, batch_size):
    train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

    train_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

    # Flow dari direktori dengan generator
    train_generator = train_datagen.flow_from_directory(
        directory= data_dir +'train',
        # subset="training",
        target_size=(img_height, img_width),
        color_mode='rgb',
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=True
    )

    # Definisikan ImageDataGenerator dengan preprocessing input
    val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

    # Flow dari direktori dengan generator
    val_generator = val_datagen.flow_from_directory(
        directory= data_dir +'val',
        # subset="validation",
        target_size=(img_height, img_width),
        color_mode='rgb',
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=True
    )

    # Definisikan ImageDataGenerator dengan preprocessing input
    test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

    # Flow dari direktori dengan generator
    test_generator = test_datagen.flow_from_directory(
        directory= data_dir +'test',
        # subset="test",
        target_size=(img_height, img_width),
        color_mode='rgb',
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False
    )

    # Dapatkan kamus nama kelas dari generator
    class_dict = train_generator.class_indices
    # Tampilkan nama-nama kelas
    class_names = list(class_dict.keys())

    return train_generator, val_generator, test_generator, class_names


def model_load(IMG_SHAPE=(224, 224, 3), class_num=13):

    # Muat model MobileNetV2 yang telah dilatih sebelumnya tanpa lapisan atas (top)
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=IMG_SHAPE)

    # Membekukan lapisan base model sehingga tidak akan dilatih ulang
    base_model.trainable = False

    # Menambahkan Global Average Pooling
    x = GlobalAveragePooling2D()(base_model.output)

    # Menambahkan lapisan Dense dengan aktivasi Softmax untuk klasifikasi 13 kelas
    output_layer = Dense(class_num, activation='softmax')(x)

    # Membuat model baru base_model = mobilenetV2(Feature Map) dan Output layer FC menggunakan softmax
    model = Model(inputs=base_model.input, outputs=output_layer)
    # Menampilkan ringkasan model
    model.summary()

    optimizer = Adam(learning_rate=0.001)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    return model


def show_loss_acc(history):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']

    loss = history.history['loss']
    val_loss = history.history['val_loss']

    plt.figure(figsize=(8, 8))
    plt.subplot(2, 1, 1)
    plt.plot(acc, label='Training Accuracy')
    plt.plot(val_acc, label='Validation Accuracy')
    plt.legend(loc='lower right')
    plt.ylabel('Accuracy')
    plt.ylim([min(plt.ylim()), 1])
    plt.title('Training and Validation Accuracy')

    plt.subplot(2, 1, 2)
    plt.plot(loss, label='Training Loss')
    plt.plot(val_loss, label='Validation Loss')
    plt.legend(loc='upper right')
    plt.ylabel('Cross Entropy')
    plt.title('Training and Validation Loss')
    plt.xlabel('epoch')
    plt.savefig('eksperimen_3.png', dpi=100)


def convert_seconds(seconds):
    # Menghitung menit dan sisa detik
    minutes = seconds // 60
    remaining_seconds = seconds % 60
    return f"{int(minutes)} menit {remaining_seconds:.2f} detik"


def train(epochs, batch_sizes):
    begin_time = time()
    train_generator, val_generator, test_generator, class_names = data_load("dataset_final/", 224, 224, 32)
    print(class_names)
    model = model_load(class_num=len(class_names))


    history = model.fit(train_generator, validation_data=val_generator, batch_size=batch_sizes, epochs=epochs,
            verbose=1)

    KERAS_MODEL_NAME = "model_epoch50_32.h5"
    model.save(KERAS_MODEL_NAME)
    convert_bytes(get_file_size(KERAS_MODEL_NAME), "MB")

    end_time = time()
    run_time = end_time - begin_time
    print('The cycle program runs for the following hours:', run_time, "s")  # The duration of the cycle program： 1.4201874732
    show_loss_acc(history)


    # Contoh penggunaan
    seconds = run_time
    time_str = convert_seconds(seconds)
    print(time_str)

    # print test
    model = tf.keras.models.load_model(KERAS_MODEL_NAME)
    model.summary()

    # test_loss, test_acc
    loss, accuracy = model.evaluate(test_generator)
    print('Mobilenev2 test accuracy :', accuracy)



    # Save training time and accuracy to CSV
    with open('hasil_eksperimen_3.csv', mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Training Time (s)', 'Test Accuracy', 'Test Loss'])
        writer.writerow([time_str , accuracy, loss])


    # conversi menjadi tflite
    TF_LITE_MODEL_FILE_NAME = "model_epoch50_32.tflite"

    tf_lite_converter = tf.lite.TFLiteConverter.from_keras_model(model)
    tflite_model = tf_lite_converter.convert()

    tflite_model_name = TF_LITE_MODEL_FILE_NAME
    open(tflite_model_name, "wb").write(tflite_model)
    convert_bytes(get_file_size(TF_LITE_MODEL_FILE_NAME), "KB")

# function untuk konversi tflite
def get_file_size(file_path):
    size = os.path.getsize(file_path)
    return size

def convert_bytes(size, unit=None):
    if unit == "KB":
        return print('File size ' + str(round(size / 1024, 3)) + 'Kilobytes')
    elif unit == "MB":
        return print('File size ' + str(round(size / 1024 * 1024, 3)) + 'Megabytes')
    else :
        return print('File size ' + str(round(size)) + 'bytes')

if __name__ == '__main__':
    train(epochs=50, batch_sizes=32)